   After each instruction finishes at its terminal state, the FSM jumps back
   to state 0 to fetch the next instruction. Each state specifies exactly which
   control signals are asserted -- for example, state 0 asserts PCWrite, IRWrite
   , MemRead (to fetch the instruction and update PC), while state 6 asserts
   MemRead and IorD=1 (to read data memory using the computed address in ALUOut)

   The FSM is implemented as combinational logic (a ROM or PLA) plus a 4-bit state
   register. The inputs are the 7-bit opcode + 4-bit current state (11 bits
   total), and the outputs are the 15 control signals + 4-bit next state (19
   bits total). It is bigger than the single-cycle control unit, but the 
   trade-off is worth it: faster average execution and less hardware duplication


THE TRADE-OFF SUMMARY   
   ... // Single-Cycle // Multi-Cycle
   Clock Period // Long (slowest instruction) // Short (slowest stage)
   Cycles per instruction // Always 1 // 3-5 depending on type
   Memory // Separate insrtr + data // Shared (reused)
   ALU // Used once // Reused across cycles
   Extra Registers // None // IR, A, B, ALUOut, MDR
   Control Unit // Combinational (simple) // FSM with state (complex)
   Average Speed // Slower // Faster

---

-- In the context of the multi-cycle datapath, `IorD` (Instruction or Data) is a
   multiplexer control signal that decides whether the memory address comes from
   the PROGRAM COUNTER (to fetch an instruction) or from the ALU OUTPUT (to
   access a data variable for `lw` or `sw`). MemRead is the command send to 
   memory telling it to putput the data at the selected address, which is active 
   during the Fetch state and the Load-Memory state. These signals are generated
   by the Control Unit, which is typically implemented as a ROM (Read-Only 
   Memory) or a PLA (Programmable Logic Array); these act as a "lookup table"
   where the input is the current state and he instruction opcode, and the 
   output is the specific pattern of 1s and 0s (the control signals) needed to
   move data through the datapath for that specific step. 

-- ... the MULTI-CYCLE DATAPATH. Instead of trying to cram everything into one
   long clock cycle (which would be slow), we break each instruction into 
   smaller steps.

   The Control Unit (FSM) coordinates this. ...


   THE "FETCH & DECODE" FOUNDATION (States 0-1)
      Every instruction, no matter what it is, starts here:

      1. STATE 0 (Fetch): The CPU fetches the instruction from memory at the 
         current PC and moves it into the Instruction Register (IR). It also
         increments the PC (PC = PC + 4).
      2. STATE 1 (Decode): The Control Unit looks at the `opcode`. While it's
         decoding, the ALU pre-calculates the branch target address just in case
         it's a branch.


   ---
   MEMORY ACCESS: `lw` and `sw` (States 2-5)
      3. STATE 2 (Mem Address): The ALU adds the base register and the offset to
         find the memory address.
      4. STATE 3 (LW Read): For `lw`, the CPU reads the data from that memory
         address.
      5. STATE 4 (LW Write-back): The data read from memory is finally saved 
         into the destination register (`rd`).
      6. STATE 5 (SW Write): For `sw`, the CPU takes the value from a register
         and writes it into the memory address calculated in State 2. 


   ---
   ARITHMETIC: R-type (States 6-7)
      7. STATE 6 (Execution): The ALU performs the actual math (e.g.,
         `x2 + x3`)     <-- `rs1 operand rs2`
      8. STATE 7 (R-type Write-back): The result of that math is stored in the
         destination regiser (`rd`).


   ---
   CONTROL FLOW: Branch && Jump
      9. STATE 8 (Branch): The ALU subtracts the two registers. If the result is
         zero (meaning they are equal), the PC is updated with the branch target
         address calculated in STATE 1.
      10. STATE 9 (JAL - Jump AND Link): The CPU saves the "return address" 
          (`PC + 4`) into a register and jumpts to the target address.
      11. STATE 10 (JALR): Similar to JAL, but the jump target is calculated by
          adding an offset to a register value (used for function returns or
          pointers).


   ---
   WHY DOES THIS MATTER FOR YOUR EXAM?
      Because different instructions take a different number of states!
         - `lw` is the "slowest" (5 states/cycles).
         - `sw` and R-type takes 4 cycles.
         - `beq` takes only 3 cycles.
                   
